# BERTopic Sentiment Integration & BPS Mapping

Notebook ini mengintegrasikan hasil BERTopic dengan hasil analisis sentimen IndoBERT.

Output:
- merged_topic_sentiment.csv
- topic_sentiment_bps.csv

In [20]:
import pandas as pd
import ast

In [21]:
from google.colab import drive
drive.mount('/content/drive')
df_topics = pd.read_csv("/content/drive/MyDrive/KaburAjaDulu/data/data_with_topics_v2.csv")
df_sentiment = pd.read_csv("/content/drive/MyDrive/KaburAjaDulu/data/sentiment_results.csv")
df_bertopic = pd.read_csv("/content/drive/MyDrive/KaburAjaDulu/data/bertopic_info_v2.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
df_merged = df_topics.merge(
    df_sentiment[["clean_text", "sentiment", "confidence"]],
    on="clean_text",
    how="left"
)
print(len(df_merged))
df_merged.head()

18814


,created_at,full_text,id_str,platform,clean_text,topic_id_v2,sentiment,confidence
0,Tue Sep 30 23:48:17 +0000 2025,EMANG BNER GUE #KaburAjaDulu ATP,1973173082253828399,X,emang bner saya kaburajadulu atp,55,negatif,99.65
1,Tue Sep 30 19:41:13 +0000 2025,Wkwkwkw masih aja. Saran saya saatnya #KaburAj...,1973110905211855020,X,wkwkwkw masih saja saran saya saatnya kaburaja...,0,netral,95.98
2,Tue Sep 30 15:25:47 +0000 2025,Selamat dan sukses kepada siswa-siswi LPK Yosh...,1973046626290897319,X,selamat dan sukses kepada siswa siswi lpk yosh...,4,positif,79.15
3,Tue Sep 30 14:43:04 +0000 2025,@alestarbluu Sumpahh aku pro ke statement #kab...,1973035876054905276,X,sumpahh aku profesional ke statement kaburajad...,0,negatif,99.58
4,Tue Sep 30 12:47:18 +0000 2025,Manual order shipped safely️ bukunya #KaburAja...,1973006742117298281,X,manual order shipped safely bukunya kaburajadu...,9,positif,81.90


Topic -1 dianggap sebagai noise karena tidak dapat dikelompokkan oleh HDBSCAN ke topik tertentu.

In [23]:
df_clean = df_merged[df_merged["topic_id_v2"] != -1].copy()

noise = df_merged[df_merged["topic_id_v2"] == -1]

print(f"Noise : {len(noise)}")
print(f"Valid : {len(df_clean)}")

Noise : 5316
Valid : 13498


In [24]:
df_clean.to_csv("merged_topic_sentiment.csv",index=False)

In [25]:
dist = (df_clean.groupby(["topic_id_v2", "sentiment"]).size().unstack(fill_value=0))

In [26]:
for col in ["positif", "netral", "negatif"]:
    if col not in dist.columns:
        dist[col] = 0

dist = dist.reset_index()

dist["total"] = (
    dist["positif"]
    + dist["netral"]
    + dist["negatif"]
)

dist["pct_negatif"] = (
    dist["negatif"] / dist["total"] * 100
).round(1)

dist["pct_positif"] = (
    dist["positif"] / dist["total"] * 100
).round(1)

dist["pct_netral"] = (
    dist["netral"] / dist["total"] * 100
).round(1)

dist["dominant_sentiment"] = (
    dist[
        ["positif", "netral", "negatif"]
    ]
    .idxmax(axis=1)
)

In [27]:
def parse_keywords(rep_str):
    try:
        kw = ast.literal_eval(rep_str)
        return ', '.join(kw[:6])
    except:
        return str(rep_str)

def clean_topic_name(name_str):
    parts = str(name_str).split('_')
    if parts[0].lstrip('-').isdigit():
        parts = parts[1:]
    return ' '.join(parts).title()

df_bertopic['keywords_clean'] = df_bertopic['Representation'].apply(parse_keywords)
df_bertopic['topic_label']    = df_bertopic['Name'].apply(clean_topic_name)

In [28]:
result = dist.merge(
    df_bertopic[
        [
            "Topic",
            "topic_label",
            "keywords_clean"
        ]
    ],
    left_on="topic_id_v2",
    right_on="Topic",
    how="left"
).drop(columns=["Topic"])

In [29]:
BPS_MAPPING = {
    # Ketenagakerjaan
    'kerja':        'Ketenagakerjaan',
    'kerjaan':      'Ketenagakerjaan',
    'pengangguran': 'Ketenagakerjaan',
    'loker':        'Ketenagakerjaan',
    'lowongan':     'Ketenagakerjaan',
    'nganggur':     'Ketenagakerjaan',
    'lulus':        'Ketenagakerjaan',
    'fresh':        'Ketenagakerjaan',
    'interview':    'Ketenagakerjaan',
    'rekrut':       'Ketenagakerjaan',
    'umur':         'Ketenagakerjaan',
    'usia':         'Ketenagakerjaan',
    # Upah & Gaji
    'gaji':         'Upah & Kesejahteraan',
    'gajinya':      'Upah & Kesejahteraan',
    'gaji jt':      'Upah & Kesejahteraan',
    'umr':          'Upah & Kesejahteraan',
    'upah':         'Upah & Kesejahteraan',
    'salary':       'Upah & Kesejahteraan',
    'sejahtera':    'Upah & Kesejahteraan',
    'duit':         'Upah & Kesejahteraan',
    'uang':         'Upah & Kesejahteraan',
    # Pendidikan
    'kuliah':       'Pendidikan',
    'kampus':       'Pendidikan',
    'mahasiswa':    'Pendidikan',
    'beasiswa':     'Pendidikan',
    'sekolah':      'Pendidikan',
    'ijasah':       'Pendidikan',
    'wisuda':       'Pendidikan',
    # Migrasi Luar Negeri
    'jepang':       'Migrasi & Ketenagakerjaan LN',
    'jerman':       'Migrasi & Ketenagakerjaan LN',
    'eropa':        'Migrasi & Ketenagakerjaan LN',
    'ausbildung':   'Migrasi & Ketenagakerjaan LN',
    'lpk':          'Migrasi & Ketenagakerjaan LN',
    'singapore':    'Migrasi & Ketenagakerjaan LN',
    'singapura':    'Migrasi & Ketenagakerjaan LN',
    'malaysia':     'Migrasi & Ketenagakerjaan LN',
    'korea':        'Migrasi & Ketenagakerjaan LN',
    'negeri':       'Migrasi & Ketenagakerjaan LN',
    'pindah':       'Migrasi & Ketenagakerjaan LN',
    # Kebijakan Pemerintah
    'pemerintah':   'Kebijakan Pemerintah',
    'pajak':        'Kebijakan Pemerintah',
    'korupsi':      'Kebijakan Pemerintah',
    'politik':      'Kebijakan Pemerintah',
    'demo':         'Kebijakan Pemerintah',
    'percaya':      'Kebijakan Pemerintah',
    'nasionalis':   'Kebijakan Pemerintah',
    'nasionalisme': 'Kebijakan Pemerintah',
    'prabowo':      'Kebijakan Pemerintah',
}

In [30]:
def map_bps(keywords_str):
    if not isinstance(keywords_str, str):
        return 'Lainnya'
    kw_lower = keywords_str.lower()
    for kw, cat in BPS_MAPPING.items():
        if kw in kw_lower:
            return cat
    return 'Lainnya'

In [31]:
result["bps_category"] = (
    result["keywords_clean"]
    .apply(map_bps)
)

result = (
    result.sort_values(
        "total",
        ascending=False
    )
    .reset_index(drop=True)
)

result["rank"] = result.index + 1

In [32]:
result.to_csv("topic_sentiment_bps.csv",index=False)

In [33]:
result[[
        "rank",
        "topic_id_v2",
        "topic_label",
        "total",
        "dominant_sentiment",
        "pct_negatif",
        "bps_category"
    ]].head(15)

,rank,topic_id_v2,topic_label,total,dominant_sentiment,pct_negatif,bps_category
0,1,0,Nasionalis Dpr Hastag Rakyat,2605,negatif,73.2,Kebijakan Pemerintah
1,2,1,Percaya Pemerintah Percaya Pemerintah Rakyat,1908,negatif,90.5,Kebijakan Pemerintah
2,3,2,Indonesia Indonesiagelap Gelap Cinta,1084,negatif,68.5,Lainnya
3,4,3,Kerja Negeri Negeri Kerja Negeri Kerja,869,negatif,48.8,Ketenagakerjaan
4,5,4,Jepang Bahasa Belajar Lpk,451,netral,35.5,Migrasi & Ketenagakerjaan LN
5,6,5,Indo Kerja Indo Indo Kerja Kerja,349,negatif,74.2,Ketenagakerjaan
6,7,6,Nasionalisme Meragukan Nasionalis Bahlil,326,negatif,79.1,Kebijakan Pemerintah
7,8,7,Gaji Jt Gajinya Bercanda,234,negatif,62.4,Upah & Kesejahteraan
8,9,8,Malaysia Singapore Malay Singapura,203,negatif,52.7,Migrasi & Ketenagakerjaan LN
9,10,9,Duit Uang Cari Uang Lulus,192,negatif,69.3,Ketenagakerjaan


In [34]:
result["bps_category"].value_counts()

,count
bps_category,
Lainnya,53
Ketenagakerjaan,11
Migrasi & Ketenagakerjaan LN,10
Upah & Kesejahteraan,9
Kebijakan Pemerintah,5
Pendidikan,2


In [35]:
result["dominant_sentiment"].value_counts()

,count
dominant_sentiment,
negatif,61
positif,18
netral,11
